# Foundation MLIPs as ASE calculators (MACE-MP & GRACE-FS)

Demonstrates that the `Engine` Protocol composes with **any** ASE calculator, including the recent universal/foundation interatomic potentials:

- [**MACE-MP**](https://github.com/ACEsuit/mace) - `mace.calculators.mace_mp(...)`
- [**GRACE-FS**](https://gracemaker.readthedocs.io) - `tensorpotential.calculator.grace_fm(...)`

Both ship ASE-compatible calculator wrappers, so the *only* line that changes between EMT, MACE, and GRACE is the `calculator=` argument to `ASEEngine`. The physics task description (FCC Al vacancy formation energy on a small supercell, below) is identical across all three backends.

For each backend we build a parent `pyiron_workflow.Workflow` that:

1. **Optimises the cubic lattice parameter** with `optimise_cubic_lattice_parameter` (cheap 5-point EOS scan).
2. **Feeds the equilibrium structure** straight into `get_vacancy_formation_energy`.

This way each foundation model picks its *own* a0 for Al, instead of inheriting a hardcoded value - the cross-FM comparison is then apples-to-apples.

If MACE / GRACE aren't installed, the corresponding cells skip with an `Install with: ...` hint so the notebook still runs end-to-end.

In [1]:
import os, warnings, logging

# Quiet down TensorFlow / e3nn / pyiron-workflow startup chatter.
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")
os.environ.setdefault("TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD", "1")
warnings.filterwarnings("ignore")
logging.disable(logging.WARNING)

from ase.build import bulk
from ase.calculators.emt import EMT

from pyiron_workflow import Workflow
from pyiron_workflow_atomistics.engine import ASEEngine, CalcInputMinimize
from pyiron_workflow_atomistics.physics.bulk import optimise_cubic_lattice_parameter
from pyiron_workflow_atomistics.physics.point_defect import get_vacancy_formation_energy

# Seed structure for the lat-opt EOS scan. The lattice parameter here is only a
# starting point - each calculator re-optimises a0 in the parent workflow below.
al_bulk = bulk("Al", "fcc", a=4.05, cubic=True)
print(f"Al bulk seed: {al_bulk.get_chemical_formula()}, n={len(al_bulk)}, a_seed=4.05 A")

Al bulk seed: Al4, n=4, a_seed=4.05 A


## Helper: parent `Workflow` = lat-opt -> vacancy-formation per calculator

The helper below wires both macros into a single parent `Workflow`. Sub-folders for each macro's
runs are kept tidy via `engine.with_working_directory(...)`. The same parent shape runs for every
backend; the only line that changes is the `calculator=` argument to `ASEEngine`.

In [2]:
def vacancy_e_form(calculator, label, workdir):
    engine = ASEEngine(
        EngineInput=CalcInputMinimize(force_convergence_tolerance=0.05, max_iterations=100),
        calculator=calculator,
        working_directory=workdir,
    )
    wf = Workflow(f"fm_{label.lower().replace('-', '_')}")
    wf.lat_opt = optimise_cubic_lattice_parameter(
        structure=al_bulk,
        name="Al",
        crystalstructure="fcc",
        engine=engine.with_working_directory("lat_opt"),
        strain_range=(-0.02, 0.02),
        num_points=5,
    )
    wf.vac = get_vacancy_formation_energy(
        structure=wf.lat_opt.outputs.equil_struct,
        engine=engine.with_working_directory("vac"),
        min_dimensions=[8, 8, 8],
    )
    wf.run()
    a0 = wf.lat_opt.outputs.a0.value
    e_f = wf.vac.outputs.vacancy_formation_energy.value
    print(f"{label:8s}  a0(Al FCC) = {a0:.4f} A   E_vac_form = {e_f:.3f} eV")
    return a0, e_f

## 1. EMT (classical, fast reference)

EMT is a coarse but-always-available potential for Al/Cu/Ni/Pd/Ag/Pt/Au. It under-predicts vacancy energies versus DFT.

In [3]:
a0_emt, e_emt = vacancy_e_form(EMT(), label="EMT", workdir="./_fm_emt")

      Step     Time          Energy          fmax
BFGS:    0 17:59:26       -0.016663        0.000000


      Step     Time          Energy          fmax
BFGS:    0 17:59:26       -0.018504        0.000000
      Step     Time          Energy          fmax
BFGS:    0 17:59:26       -0.006008        0.000000
      Step     Time          Energy          fmax
BFGS:    0 17:59:26        0.020062        0.000000
      Step     Time          Energy          fmax
BFGS:    0 17:59:26        0.058934        0.000000
      Step     Time          Energy          fmax
BFGS:    0 17:59:26        0.453946        0.103855
BFGS:    1 17:59:26        0.451677        0.098047


BFGS:    2 17:59:26        0.433813        0.024509
      Step     Time          Energy          fmax
BFGS:    0 17:59:27       -0.527330        0.000000
EMT       a0(Al FCC) = 3.9942 A   E_vac_form = 0.956 eV


## 2. MACE-MP (universal foundation model)

`mace_mp(model="small")` downloads the universal MACE-MP-0 small foundation potential on first use (~30 MB) and caches it under `~/.cache/mace/`. Trained on Materials Project — works for any element combination in MP.

Install: `pip install mace-torch` (or `uv pip install mace-torch`).

In [4]:
try:
    from mace.calculators import mace_mp
    mace_calc = mace_mp(model="small", default_dtype="float64", device="cpu")
    a0_mace, e_mace = vacancy_e_form(mace_calc, label="MACE-MP", workdir="./_fm_mace")
except ImportError:
    a0_mace, e_mace = None, None
    print("MACE not installed - skipping.\nInstall with: pip install mace-torch  (or `uv pip install mace-torch`)")

MACE not installed - skipping.
Install with: pip install mace-torch  (or `uv pip install mace-torch`)


## 3. GRACE-FS (universal foundation model)

`grace_fm("GRACE-FS-OAM")` downloads the GRACE-FS foundation potential on first use and caches it under `~/.cache/grace/`. Trained on OMat24 + sAlex + MPTraj — typically faster than MACE-MP on CPU thanks to a compact graph formulation.

Install: `pip install tensorpotential` (or `uv pip install tensorpotential`).

In [5]:
try:
    from tensorpotential.calculator import grace_fm
    grace_calc = grace_fm("GRACE-FS-OAM")
    a0_grace, e_grace = vacancy_e_form(grace_calc, label="GRACE-FS", workdir="./_fm_grace")
except ImportError:
    a0_grace, e_grace = None, None
    print("GRACE / tensorpotential not installed - skipping.\nInstall with: pip install tensorpotential  (or `uv pip install tensorpotential`)")

GRACE / tensorpotential not installed - skipping.
Install with: pip install tensorpotential  (or `uv pip install tensorpotential`)


## Summary

Same parent `Workflow` shape (lat-opt -> vacancy-formation), same seed structure, three calculators. Each foundation MLIP picks its own equilibrium a0 for FCC Al before the supercell is built, so the formation energy reflects that backend's own lattice. Foundation MLIPs (MACE, GRACE) target DFT-PBE quality (~0.6 eV for Al vacancy in PBE); EMT systematically under-predicts.

To plug in your own potential - Quip GAP, NequIP, Allegro, an ASE-wrapped DFT code, anything - just hand its ASE calculator to `ASEEngine(calculator=...)` and reuse the rest.

In [6]:
print(f"{'backend':10s} {'a0 (A)':>10s} {'E_vac (eV)':>12s}")
print("-" * 34)
for label, a0, ev in [
    ("EMT", a0_emt, e_emt),
    ("MACE-MP", a0_mace, e_mace),
    ("GRACE-FS", a0_grace, e_grace),
]:
    a0_s = f"{a0:.4f}" if a0 is not None else "skipped"
    ev_s = f"{ev:.4f}" if ev is not None else "skipped"
    print(f"{label:10s} {a0_s:>10s} {ev_s:>12s}")

backend        a0 (A)   E_vac (eV)
----------------------------------
EMT            3.9942       0.9563
MACE-MP       skipped      skipped
GRACE-FS      skipped      skipped
